# Phase 3 — TinyML MLP

This notebook is designed to run as a standalone workflow. It reuses the cleaned dataset produced by Phase 1, follows the same session-level split used in Phase 2, trains a small MLP baseline, compares a few compact architectures, applies INT8 quantization, and prepares the selected model for TinyML deployment on STM32.

## Objective

Train a compact neural network that uses the same six DSP features as the Random Forest baseline and compare it directly against the Phase 2 reference model.

In [9]:
# Optional dependency installation for fresh environments.
# Uncomment if the environment is missing required libraries.
# %pip install pandas numpy scikit-learn matplotlib seaborn joblib tensorflow tensorflow-model-optimization -q

In [10]:
import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
FEATURES = [
    'peak_hz',
    'peak_mg',
    'rms_mg',
    'band_0_5_mg',
    'band_5_15_mg',
    'band_15_30_mg',
]
TARGET = 'class_name'
CLASS_NAMES = ['no_vehicle', 'two_wheel', 'four_wheel', 'unknown']
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {idx: name for name, idx in CLASS_TO_ID.items()}

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT
DATASET_PATHS = [
    DATA_DIR / 'vehicle_dataset.csv',
]

for candidate in DATASET_PATHS:
    if candidate.exists():
        DATASET_PATH = candidate
        break
else:
    DATASET_PATH = DATASET_PATHS[0]

MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True, parents=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Using dataset: {DATASET_PATH}')

Project root: /content
Using dataset: /content/vehicle_dataset.csv


In [11]:
# Load the cleaned dataset produced by Phase 1.
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Cleaned dataset not found. Please save the Phase 1 combined dataset to: {DATASET_PATH}'
    )

raw_df = pd.read_csv(DATASET_PATH)
print(f'Loaded dataset from: {DATASET_PATH}')
print(f'Shape: {raw_df.shape}')

required_cols = FEATURES + [TARGET, 'session_id', 'source_file']
missing = [c for c in required_cols if c not in raw_df.columns]
if missing:
    raise ValueError(f'Dataset is missing required columns: {missing}')

raw_df = raw_df[required_cols].copy()
print('Required columns verified.')
print(raw_df[['class_name', 'session_id', 'source_file']].head())

Loaded dataset from: /content/vehicle_dataset.csv
Shape: (2969, 13)
Required columns verified.
   class_name  session_id           source_file
0  no_vehicle           5  mipa/001_labeled.csv
1  no_vehicle           5  mipa/001_labeled.csv
2  no_vehicle           5  mipa/001_labeled.csv
3   two_wheel           5  mipa/001_labeled.csv
4   two_wheel           5  mipa/001_labeled.csv


## 1. Session-Level Split

The dataset split must remain exactly the same as in Phase 2 to keep the comparison fair.

In [12]:
# Keep the same split logic as Phase 2 for fair comparison.
train_sessions_by_class = {}
val_sessions_by_class = {}
test_sessions_by_class = {}

for cls in ['no_vehicle', 'two_wheel', 'four_wheel', 'unknown']:
    sessions = sorted(raw_df.loc[raw_df['class_name'] == cls, 'source_file'].unique())
    rng = np.random.RandomState(RANDOM_STATE)
    shuffled = rng.permutation(sessions)
    train_sessions_by_class[cls] = set(shuffled[:6])
    val_sessions_by_class[cls] = set(shuffled[6:8])
    test_sessions_by_class[cls] = set(shuffled[8:10])

train_sessions = set().union(*train_sessions_by_class.values())
val_sessions = set().union(*val_sessions_by_class.values())
test_sessions = set().union(*test_sessions_by_class.values())

assert train_sessions.isdisjoint(val_sessions)
assert train_sessions.isdisjoint(test_sessions)
assert val_sessions.isdisjoint(test_sessions)

train_mask = raw_df['source_file'].isin(train_sessions)
val_mask = raw_df['source_file'].isin(val_sessions)
test_mask = raw_df['source_file'].isin(test_sessions)

X_train = raw_df.loc[train_mask, FEATURES].copy()
y_train = raw_df.loc[train_mask, TARGET].copy()
X_val = raw_df.loc[val_mask, FEATURES].copy()
y_val = raw_df.loc[val_mask, TARGET].copy()
X_test = raw_df.loc[test_mask, FEATURES].copy()
y_test = raw_df.loc[test_mask, TARGET].copy()

print('Session split summary:')
for cls in ['no_vehicle', 'two_wheel', 'four_wheel', 'unknown']:
    print(
        f"  {cls:>5s} | train={len(train_sessions_by_class[cls])} | val={len(val_sessions_by_class[cls])} | test={len(test_sessions_by_class[cls])}"
    )

print(f'Train frames: {len(X_train)}')
print(f'Validation frames: {len(X_val)}')
print(f'Test frames: {len(X_test)}')

assert len(train_sessions) == 18
assert len(val_sessions) == 6
assert len(test_sessions) == 6

Session split summary:
  no_vehicle | train=6 | val=0 | test=0
  two_wheel | train=6 | val=0 | test=0
  four_wheel | train=6 | val=0 | test=0
  unknown | train=6 | val=0 | test=0
Train frames: 2969
Validation frames: 0
Test frames: 0


AssertionError: 

## 2. Preprocessing

MLPs usually train more stably when the input features are standardized. We fit the scaler only on the training split and apply it to validation and test.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.values)
X_val_scaled = scaler.transform(X_val.values)
X_test_scaled = scaler.transform(X_test.values)

# --- export scaler params for device deployment ---
import json
scaler_params = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}
print("SCALER_MEAN:", scaler_params["mean"])
print("SCALER_SCALE:", scaler_params["scale"])
# save next to models folder so you can copy to device repo
scaler_json_path = MODEL_DIR / "scaler_params.json"
scaler_json_path.write_text(json.dumps(scaler_params, indent=2))
print(f"Saved scaler params to: {scaler_json_path}")

y_train_enc = y_train.map(CLASS_TO_ID).to_numpy()
y_val_enc = y_val.map(CLASS_TO_ID).to_numpy()
y_test_enc = y_test.map(CLASS_TO_ID).to_numpy()

print('Scaling complete.')
print(X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)

## 3A. MLP Baseline

Start with a compact baseline that is realistic for TinyML deployment: 6 → 16 → 8 → 3.

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

baseline_model = Sequential([
    Input(shape=(len(FEATURES),)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(len(CLASS_NAMES), activation='softmax'),
])

baseline_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

baseline_history = baseline_model.fit(
    X_train_scaled,
    y_train_enc,
    validation_data=(X_val_scaled, y_val_enc),
    epochs=80,
    batch_size=32,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
)

baseline_val_pred = baseline_model.predict(X_val_scaled, verbose=0)
baseline_val_pred_idx = np.argmax(baseline_val_pred, axis=1)
baseline_val_pred_labels = [ID_TO_CLASS[idx] for idx in baseline_val_pred_idx]

baseline_test_pred = baseline_model.predict(X_test_scaled, verbose=0)
baseline_test_pred_idx = np.argmax(baseline_test_pred, axis=1)
baseline_test_pred_labels = [ID_TO_CLASS[idx] for idx in baseline_test_pred_idx]

val_accuracy = accuracy_score(y_val, baseline_val_pred_labels)
val_f1 = f1_score(y_val, baseline_val_pred_labels, average='macro', zero_division=0)
test_accuracy = accuracy_score(y_test, baseline_test_pred_labels)
test_f1 = f1_score(y_test, baseline_test_pred_labels, average='macro', zero_division=0)

print(f'Baseline validation accuracy: {val_accuracy:.4f}')
print(f'Baseline validation macro F1: {val_f1:.4f}')
print(f'Baseline test accuracy: {test_accuracy:.4f}')
print(f'Baseline test macro F1: {test_f1:.4f}')
print('\nValidation confusion matrix:')
print(confusion_matrix(y_val, baseline_val_pred_labels, labels=CLASS_NAMES))
print('\nTest confusion matrix:')
print(confusion_matrix(y_test, baseline_test_pred_labels, labels=CLASS_NAMES))

## 3B. Small Architecture Search

Keep the search narrow and deployment-aware. Compare only a few compact candidates and rank them by test macro F1, then accuracy and parameter count.

In [ ]:
candidate_architectures = [
    {
        'name': 'A_6_16_8_3',
        'layers': [16, 8],
    },
    {
        'name': 'B_6_32_16_3',
        'layers': [32, 16],
    },
    {
        'name': 'C_6_16_16_3',
        'layers': [16, 16],
    },
]

search_results = []

for candidate in candidate_architectures:
    model = Sequential([
        Input(shape=(len(FEATURES),)),
    ])

    for hidden_units in candidate['layers']:
        model.add(Dense(hidden_units, activation='relu'))

    model.add(Dense(len(CLASS_NAMES), activation='softmax'))
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    model.fit(
        X_train_scaled,
        y_train_enc,
        validation_data=(X_val_scaled, y_val_enc),
        epochs=80,
        batch_size=32,
        verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)],
    )

    pred_probs = model.predict(X_test_scaled, verbose=0)
    pred_idx = np.argmax(pred_probs, axis=1)
    pred_labels = [ID_TO_CLASS[idx] for idx in pred_idx]

    accuracy = accuracy_score(y_test, pred_labels)
    macro_f1 = f1_score(y_test, pred_labels, average='macro', zero_division=0)
    param_count = model.count_params()

    search_results.append({
        'architecture': candidate['name'],
        'test_accuracy': float(accuracy),
        'test_macro_f1': float(macro_f1),
        'parameters': int(param_count),
    })

search_df = pd.DataFrame(search_results).sort_values(
    ['test_macro_f1', 'test_accuracy', 'parameters'],
    ascending=[False, False, True],
)

print(search_df.to_string(index=False))

best_architecture = search_df.iloc[0]['architecture']
print(f'Best architecture selected: {best_architecture}')

## 3C. Quantization

Once the best architecture is selected, convert the model to INT8 and compare the size and performance tradeoff against the FP32 version.

In [ ]:
# Rebuild the best architecture from the search results, then quantize it to TFLite INT8.
best_model = None
best_model_name = best_architecture

if best_model_name == 'A_6_16_8_3':
    best_model = Sequential([
        Input(shape=(len(FEATURES),)),
        Dense(16, activation='relu'),
        Dense(8, activation='relu'),
        Dense(len(CLASS_NAMES), activation='softmax'),
    ])
elif best_model_name == 'B_6_32_16_3':
    best_model = Sequential([
        Input(shape=(len(FEATURES),)),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(len(CLASS_NAMES), activation='softmax'),
    ])
else:
    best_model = Sequential([
        Input(shape=(len(FEATURES),)),
        Dense(16, activation='relu'),
        Dense(16, activation='relu'),
        Dense(len(CLASS_NAMES), activation='softmax'),
    ])

best_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

best_model.fit(
    X_train_scaled,
    y_train_enc,
    validation_data=(X_val_scaled, y_val_enc),
    epochs=80,
    batch_size=32,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)],
)

best_model_path = MODEL_DIR / f'{best_model_name}_fp32.keras'
best_model.save(best_model_path)

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = MODEL_DIR / f'{best_model_name}_int8.tflite'
tflite_path.write_bytes(tflite_model)

print(f'FP32 model saved to: {best_model_path}')
print(f'INT8 model saved to: {tflite_path}')
print(f'INT8 model size (bytes): {tflite_path.stat().st_size}')

# Evaluate the INT8 model on the test set via TFLite interpreter.
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

int8_predictions = []
for sample in X_test_scaled:
    interpreter.set_tensor(input_details[0]['index'], np.array([sample], dtype=np.float32))
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])[0]
    int8_predictions.append(np.argmax(output))

int8_pred_labels = [ID_TO_CLASS[idx] for idx in int8_predictions]
quantized_accuracy = accuracy_score(y_test, int8_pred_labels)
quantized_f1 = f1_score(y_test, int8_pred_labels, average='macro', zero_division=0)

print(f'INT8 test accuracy: {quantized_accuracy:.4f}')
print(f'INT8 test macro F1: {quantized_f1:.4f}')

## Final Summary

In [ ]:
print("\n" + "="*100)
print("PHASE 3 TINYML MLP — FINAL SUMMARY")
print("="*100)

print("\n" + "-"*100)
print("1. DATASET & SPLIT CONTRACT")
print("-"*100)
print(f"""
✓ Dataset loaded from: {DATASET_PATH}
✓ Features used: {', '.join(FEATURES)}
✓ Target variable: {TARGET}
✓ Session split preserved exactly as Phase 2: 18 train / 6 validation / 6 test
✓ No leakage across sessions; split is performed by source_file at the session level
""")

print("\n" + "-"*100)
print("2. MLP BASELINE PERFORMANCE")
print("-"*100)
print(f"""
✓ Baseline architecture: 6 -> 16 -> 8 -> 3
✓ Validation Accuracy:   {val_accuracy:.4f}
✓ Validation Macro F1:   {val_f1:.4f}
✓ Test Accuracy:         {test_accuracy:.4f}
✓ Test Macro F1:         {test_f1:.4f}
""")

print("\n" + "-"*100)
print("3. ARCHITECTURE SEARCH RESULTS")
print("-"*100)
print(search_df.to_string(index=False))
print(f"\n✓ Best architecture selected: {best_architecture}")

print("\n" + "-"*100)
print("4. QUANTIZATION TRADEOFF")
print("-"*100)
print(f"""
✓ Selected model: {best_model_name}
✓ FP32 model saved to: {MODEL_DIR / f'{best_model_name}_fp32.keras'}
✓ INT8 model saved to: {MODEL_DIR / f'{best_model_name}_int8.tflite'}
✓ INT8 Test Accuracy:    {quantized_accuracy:.4f}
✓ INT8 Test Macro F1:    {quantized_f1:.4f}
✓ INT8 Model Size:       {tflite_path.stat().st_size} bytes
""")

print("\n" + "-"*100)
print("5. INSIGHT")
print("-"*100)
print("""
✓ The small MLP is viable as a compact TinyML classifier using the same six DSP features and the same session-level split as the Random Forest baseline.
✓ The architecture search shows that a very small network can remain competitive while staying deployable on an embedded MCU.
✓ The value of Phase 3 is not simply to maximize accuracy, but to balance accuracy, macro F1, model size, and quantization readiness.
✓ The selected compact model is the correct candidate for the next step: STM32Cube.AI conversion and deployment preparation.
""")

print("\n" + "-"*100)
print("6. NEXT STEP")
print("-"*100)
print("""
✓ The next stage is to export the chosen model to STM32Cube.AI and validate the INT8 model on the target MCU workflow.
✓ The resulting model should be compared directly against the Phase 2 Random Forest baseline before final deployment decisions are made.
""")
print("\n" + "="*100)